# Assorted dataset queries

## Setup

In [ ]:
from pathlib import Path
from pprint import pprint

# IN _env.py:
#
# from pathlib import Path
# test_bids_read_path=Path("/path/to/BIDS_out")
from _env import test_bids_read_path

from clinicaio import BIDSDataset, DataType, ImageQuery, Session

dataset = BIDSDataset.populate_from_dir(
    bids_dir=test_bids_read_path,
    subjects_info=False,
    sessions_info=False,
    # broken scans.tsv
    image_scans_info=False,
)

for image in dataset.all_images():
    print(image)

## Full ImageQuery

In [ ]:
image_query = ImageQuery(
    subjects=["sub-ADNI027S0074"],
    sessions=["ses-M000"],
    data_type=DataType.PET,
    entities={"trc": "18FFDG", "rec": "coregiso8"},
    suffix="pet",
)
list(dataset.query_images_nifti_paths(image_query))

## QUERIES

## Give me all tsv files

In [ ]:
[subject.info for subject in dataset.all_subjects() if not subject.info.is_empty()]

In [ ]:
[session.info for session in dataset.all_sessions() if not session.info.is_empty()]

In [ ]:
[image.scan_info for image in dataset.all_images() if not image.scan_info.is_empty()]

## Give me all images with this tracer (ex 18FFDG)

In [ ]:
images = dataset.query_images(ImageQuery(entities={"trc": "18FFDG"}))
# p.ex
for image in images:
    print(image.get_nifti_image_path(), image)

## Give me all T1w images paths

In [ ]:
list(dataset.query_images_nifti_paths(ImageQuery(suffix="T1w")))

## Give me all modalities for this one subject

In [ ]:
# subject_id = "sub-ADNI027S0074"
subject_id = "sub-001"
# subject_id = "sub-AIBL1455"
print(subject_id)
subject = dataset.subject_by_id(subject_id)
assert subject is not None
set(image.suffix for image in subject.all_images())

## Give me all sessions for this one subject

In [ ]:
print(subject_id)
pprint(list(subject.all_sessions()))

## Give me all subjects/sessions that have both a T1 and a PET image for the same session

In [ ]:
def has_t1_and_pet(session: Session):
    has_t1 = any(image.suffix == "T1w" for image in session.all_images())

    return (
        has_t1 and next(iter(session.images_by_data_type(DataType.PET)), None) is None
    )


subjects_and_sessions_with_t1_and_pet = filter(
    has_t1_and_pet,
    dataset.all_sessions(),
)
for session in subjects_and_sessions_with_t1_and_pet:
    subject = session.parent_subject
    print(subject.id)
    pprint(session)
    print("========================")

## Give me the subjects that have more than one session

In [ ]:
pprint(
    list(filter(lambda subject: subject.sessions_count() > 1, dataset.all_subjects()))
)

## Check that all subjects/sessions have FLAIR images

In [ ]:
def session_has_flair_image(session: Session):
    any(image.suffix == "FLAIR" for image in session.all_images())


all(session_has_flair_image(session) for session in dataset.all_sessions())

## Give me all the modalities available for each subject for this list of subjects

In [ ]:
subjects = dataset.all_subjects()

{subject.id: {image.suffix for image in subject.all_images()} for subject in subjects}

## Give me all the modalities available for all subjects for this list of subjects

In [ ]:
print(
    {
        f"{image.suffix}"
        for subject in dataset.all_subjects()
        for image in subject.all_images()
    }
)

## Can you tell me if all subjects have only one session

In [ ]:
all(subject.sessions_count() == 1 for subject in dataset.all_subjects())

In [ ]:
# Roundtrip testing

import os
import shutil

print("WRITING COPY OF DATASET")
dataset.bids_path = Path("/tmp/bids_foobar_write")
shutil.rmtree(dataset.bids_path, ignore_errors=True)
for session in dataset.all_sessions():
    images = list(session.all_images())

    session._images = {}
    for image in images:
        added_image = session.write_image(
            data_type=image.data_type,
            nifti_extension=image.nifti_extension,
            entities=image.entities,
            suffix=image.suffix,
            scan_info=image.scan_info,
        )
        nifti_path = added_image.get_nifti_image_path()
        with open(nifti_path, mode="x") as f:
            print("NIFTI IMAGE", file=f)

dataset.write_to_folder(readme="DATASET COPY README")

dataset2 = BIDSDataset.populate_from_dir(
    dataset.bids_path, sessions_info=True, subjects_info=True, image_scans_info=True
)
# print(dataset2.__repr__(), file=open("aaaa", mode="w"))
# print(dataset.__repr__(), file=open("bbbb", mode="w"))
assert dataset2 == dataset
print("datasets are identically after round-trip")